## Running Abil

In [3]:
#paths:
from pathlib import Path
#handling data:
import pandas as pd
from yaml import load
from yaml import CLoader as Loader
from datetime import datetime
#abil functions:
from abil.tune import ModelTuner as tune
from abil.predict import ModelPredictor as predict
from abil.post import AbilPostProcessor as post


In [4]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()
conffile = PROJECT_ROOT / "1-phase example" / "regressor.yml"

# Load model configuration
with conffile.open('r') as f:
    model_config = load(f, Loader=Loader)

model_config['root'] = str(PROJECT_ROOT) + '/'
model_config['local_root'] = str(PROJECT_ROOT) + '/'


In [5]:
# Load training data
targets = pd.read_csv(PROJECT_ROOT / model_config['targets'])
d = pd.read_csv(PROJECT_ROOT / model_config['training'])

# Define target
target = targets['Target'][0]

# Define predictors based on YAML
predictors = model_config['predictors']
d = d.dropna(subset=predictors)

# Split training data into X_train and y
y = d[target]
X_train = d[predictors]

print("finished loading data")


finished loading data


In [ ]:
#setup model:
m = tune(X_train, y, model_config)

#run model:
m.train(model='rf')
m.train(model='knn')
#m.train(model='xgb') # skipping to save time


beginning init
length of y: 34612
training regressor
{'regressor__estimator__n_estimators': [100], 'regressor__estimator__max_features': [0.8], 'regressor__estimator__max_depth': [51], 'regressor__estimator__min_samples_leaf': [1], 'regressor__estimator__max_samples': [0.9]}
Fitting 10 folds for each of 1 candidates, totalling 10 fits
exported model to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/model/rf/Prochlorococcus_reg.sav


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    7.3s remaining:   29.3s


exported scoring to:  /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/scoring/rf/Prochlorococcus_reg.sav
reg rRMSE: 148%
reg rMAE: 60%
reg R2: 0.60
execution time: 0.0 seconds
training regressor
{'regressor__estimator__max_samples': [0.9], 'regressor__estimator__max_features': [0.8], 'regressor__estimator__estimator__leaf_size': [30], 'regressor__estimator__estimator__n_neighbors': [3], 'regressor__estimator__estimator__p': [1], 'regressor__estimator__estimator__weights': ['distance']}


[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    8.9s finished


Fitting 10 folds for each of 1 candidates, totalling 10 fits
exported model to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/model/knn/Prochlorococcus_reg.sav


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.9s remaining:    3.5s


... (2 outputs left out)

Fitting 10 folds for each of 2 candidates, totalling 20 fits
exported model to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/model/xgb/Prochlorococcus_reg.sav


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    2.2s remaining:    8.9s


exported scoring to:  /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/scoring/xgb/Prochlorococcus_reg.sav
reg rRMSE: 153%
reg rMAE: 62%
reg R2: 0.56
execution time: 0.0 seconds


[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    2.9s finished


In [ ]:
# Load prediction data
X_predict = pd.read_csv(PROJECT_ROOT / model_config['prediction'])
X_predict.set_index(["time", "depth", "lat", "lon"], inplace=True)
X_predict = X_predict[predictors]

# Setup model
m = predict(X_train, y, X_predict, model_config)

# Predict model
m.make_prediction()


In [ ]:
target_names = targets['Target'].values
target_subset = target_names[:1] # subset for estimate_applicability and merge_obs
current_date = datetime.today().strftime('%Y-%m-%d')

def do_post(statistic):
    m = post(X_train, y, X_predict, model_config, statistic)
    
    #if statistic == "mean": # Skipping to save time
        #print('begin estimate_applicability')
        #m.estimate_applicability(target_subset)

    print('begin export_ds')
    m.export_ds(current_date)

    print('begin integration')
    magnitude_conversion = 1e-21
    molar_mass = 12.01
    integ = m.integration(m, magnitude_conversion=magnitude_conversion,molar_mass=molar_mass,rate=True)
    if statistic == "mean":
        integ.integrated_totals(target_names)
    else:
        integ.integrated_totals(target_subset)

    print('do_post for: ', statistic, ' complete')

In [8]:
do_post(statistic="mean")
do_post(statistic="ci95_UL")
do_post(statistic="ci95_LL")

merging... /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/predictions/ens
finished merging NetCDF files
Model configuration exported to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_surf_reg/posts/model_config.yml
the target is: Prochlorococcus
finished merging parameters
the target is: Prochlorococcus
finished merging parameters
the target is: Prochlorococcus
finished merging parameters
models included in merge performance! ['rf', 'knn', 'xgb']
finished merging performance
finished merging performance
finished merging performance
finished merging performance
begin export_ds
export_ds
dataframe:                      Prochlorococcus
time depth lat lon                  
4    0     -78  165         2.527536
                166         2.415043
           -77 -175         0.056284
               -174         0.058176
               -173         0.059885
                     Prochlorococcus
time depth lat lon                  
4    0     -78  165         2.527536
  